# C Black-Scholes Pricing Kernel

This notebook compiles and validates a small C pricing kernel for the Black-Scholes model. Python remains responsible for data handling, analysis, plotting, and project workflow. The C library is used for selected numerical functions.

## Implemented C functions

| Function | Purpose |
|---|---|
| `normal_pdf` | Standard normal density |
| `normal_cdf` | Standard normal cumulative probability using `erf` |
| `bs_d1` | Black-Scholes \(d_1\) |
| `bs_d2` | Black-Scholes \(d_2\) |
| `bs_call_price` | European call price |
| `bs_put_price` | European put price |

The C functions use `double`, which is the standard floating-point type for this level of numerical pricing work.

In [ ]:
from pathlib import Path
import platform
import shutil
import subprocess
import sys
import timeit

import pandas as pd

In [ ]:
project_root = Path.cwd()

if not (project_root / "c_src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

## Compile the shared library

The compile command depends on the operating system. The compiled output is written to the `compiled` directory.

In [ ]:
compiled_dir = project_root / "compiled"
compiled_dir.mkdir(exist_ok=True)

compiler = shutil.which("gcc") or shutil.which("clang")
if compiler is None:
    raise RuntimeError("No C compiler found. Install gcc or clang before running this notebook.")

system_name = platform.system().lower()

if system_name == "darwin":
    library_path = compiled_dir / "libbs_pricer.dylib"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-dynamiclib",
        str(project_root / "c_src" / "normal_math.c"),
        str(project_root / "c_src" / "bs_pricer.c"),
        "-o",
        str(library_path),
    ]
elif system_name == "windows":
    library_path = compiled_dir / "bs_pricer.dll"
    compile_command = [
        compiler,
        "-O3",
        "-shared",
        str(project_root / "c_src" / "normal_math.c"),
        str(project_root / "c_src" / "bs_pricer.c"),
        "-o",
        str(library_path),
    ]
else:
    library_path = compiled_dir / "libbs_pricer.so"
    compile_command = [
        compiler,
        "-O3",
        "-fPIC",
        "-shared",
        str(project_root / "c_src" / "normal_math.c"),
        str(project_root / "c_src" / "bs_pricer.c"),
        "-o",
        str(library_path),
        "-lm",
    ]

subprocess.run(compile_command, check=True)
library_path

## Load the C library from Python

The `ctypes` wrapper defines each C function's argument types and return type before calling it from Python.

In [ ]:
from src.black_scholes import call_price, put_price
from src.c_bindings import BlackScholesCLibrary

bs_c = BlackScholesCLibrary(library_path)

## Python versus C comparison

The same option inputs are priced with both implementations. The absolute difference should be close to floating-point rounding error.

In [ ]:
S = 100.0
K = 100.0
T = 0.5
r = 0.04
sigma = 0.25

comparison = pd.DataFrame(
    {
        "option_type": ["call", "put"],
        "python_price": [
            call_price(S, K, T, r, sigma),
            put_price(S, K, T, r, sigma),
        ],
        "c_price": [
            bs_c.call_price(S, K, T, r, sigma),
            bs_c.put_price(S, K, T, r, sigma),
        ],
    }
)

comparison["absolute_difference"] = (
    comparison["python_price"] - comparison["c_price"]
).abs()

comparison.round(10)

## Small benchmark

This benchmark compares repeated scalar pricing calls. It is a local implementation check, not a general performance claim.

In [ ]:
iterations = 100_000

python_call_time = timeit.timeit(
    lambda: call_price(S, K, T, r, sigma),
    number=iterations,
)

c_call_time = timeit.timeit(
    lambda: bs_c.call_price(S, K, T, r, sigma),
    number=iterations,
)

benchmark = pd.DataFrame(
    {
        "implementation": ["Python", "C through ctypes"],
        "iterations": [iterations, iterations],
        "total_seconds": [python_call_time, c_call_time],
        "microseconds_per_call": [
            python_call_time / iterations * 1_000_000,
            c_call_time / iterations * 1_000_000,
        ],
    }
)

benchmark.round(4)

## Performance interpretation

C can be faster when repeated numerical loops are moved into compiled code. A scalar function called many times through `ctypes` may not always be faster because each call crosses the Python-to-C boundary.

Any performance statement should be based on benchmark results from the implementation being used.